## AF3에서 residue 개수 gt와 맞춰주기 

In [1]:
import os

af3_dir = '/home/psh/af3_output_processed'
gt_dir = '/home/psh/benchmark_after210930/pdb'


def get_residues_from_pdb(pdb_file):
    residues = set()
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                chain_id = line[21]
                res_id = int(line[22:26])
                residues.add((chain_id, res_id))
    return residues

def filter_pdb_by_residues(input_pdb, allowed_residues, output_pdb):
    with open(input_pdb, 'r') as fin, open(output_pdb, 'w') as fout:
        for line in fin:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                chain_id = line[21]
                res_id = int(line[22:26])
                if (chain_id, res_id) in allowed_residues:
                    fout.write(line)
            elif line.startswith("TER") or line.startswith("END"):
                fout.write(line)

for pdb in os.listdir(af3_dir):
    pdb_path = os.path.join(af3_dir, pdb)
    gt_pdb_path = os.path.join(gt_dir, pdb + '.pdb')
    if not os.path.isfile(gt_pdb_path):
        print(f"GT PDB not found for {pdb}")
        continue

    allowed_residues = get_residues_from_pdb(gt_pdb_path)
    for sample in os.listdir(pdb_path):
        sample_path = os.path.join(pdb_path, sample, "sample_1.pdb")
        if not os.path.isfile(sample_path):
            print(f"Sample PDB not found: {sample_path}")
            continue
    
        output_pdb = os.path.join(pdb_path, sample, f"{pdb}_filtered.pdb")
        filter_pdb_by_residues(sample_path, allowed_residues, output_pdb)
        print(f"Filtered PDB saved: {output_pdb}")

Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_0/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_1/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_10/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_11/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_12/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_13/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_14/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_15/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_16/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_processed/7df1_F_J_C/sample_17/7df1_F_J_C_filtered.pdb
Filtered PDB saved: /home/psh/af3_output_p

## 예측된 샘플들을 모두 pred_samples로 옮기기 (af3)

In [5]:
import os 
import shutil 

output_dir = '/home/psh/protein-frame-flow/pred_samples'
my_dir = '/home/psh/protein-frame-flow/inference_outputs/fm_tri_aa_nb_sample_x2_diff_crop/2025-05-08_19-10-51/epoch=75-step=107616/run_2025-05-10_22-53-09'
af3_dir = '/home/psh/af3_output_processed'

for pdb in os.listdir(af3_dir):
    pdb_path = os.path.join(af3_dir, pdb)
    if not os.path.isfile(gt_pdb_path):
        print(f"GT PDB not found for {pdb}")
        continue
    
    for sample in os.listdir(pdb_path):
        sample_path = os.path.join(pdb_path, sample, f"{pdb}_filtered.pdb")

        output_pdb_dir = os.path.join(output_dir, pdb)
        if not os.path.exists(output_pdb_dir):
            os.makedirs(output_pdb_dir, exist_ok=True)
        output_path = os.path.join(output_pdb_dir, f'af3_{sample}.pdb')
        shutil.copy(sample_path, output_path)

### cdr rmsd (json) 옮기기 

In [10]:
import os 
import shutil 

output_dir = '/home/psh/protein-frame-flow/pred_samples'
af3_dir = '/home/psh/af3_output_processed'

for pdb in os.listdir(af3_dir):
    pdb_path = os.path.join(af3_dir, pdb)
    if not os.path.isfile(gt_pdb_path):
        print(f"GT PDB not found for {pdb}")
        continue
    
    for sample in os.listdir(pdb_path):
        sample_path = os.path.join(pdb_path, sample, f"cdr_rmsd.json")

        output_pdb_dir = os.path.join(output_dir, pdb)
        if not os.path.exists(output_pdb_dir):
            os.makedirs(output_pdb_dir, exist_ok=True)
        output_path = os.path.join(output_pdb_dir, f'af3_{sample}.json')
        shutil.copy(sample_path, output_path)

## 예측된 샘플들을 모두 pred_samples로 옮기기 (my model)

In [9]:
import os 
import shutil 

output_dir = '/home/psh/protein-frame-flow/pred_samples'
my_dir = '/home/psh/protein-frame-flow/inference_outputs/fm_tri_aa_nb_sample_x2_diff_crop/2025-05-08_19-10-51/epoch=75-step=107616/run_2025-05-10_22-53-09'

for pdb in os.listdir(my_dir):
    if not '.' in pdb:
        pdb_path = os.path.join(my_dir, pdb)
        if not os.path.isfile(gt_pdb_path):
            print(f"GT PDB not found for {pdb}")
            continue
    
        for sample in os.listdir(pdb_path):
            sample_path = os.path.join(pdb_path, sample, f"sample_1.pdb")

            output_pdb_dir = os.path.join(output_dir, pdb)
            if not os.path.exists(output_pdb_dir):
                os.makedirs(output_pdb_dir, exist_ok=True)
            output_path = os.path.join(output_pdb_dir, f'my_{sample}.pdb')
            shutil.copy(sample_path, output_path)

### cdr rmsd (json) 옮기기 

In [11]:
import os 
import shutil 

output_dir = '/home/psh/protein-frame-flow/pred_samples'
my_dir = '/home/psh/protein-frame-flow/inference_outputs/fm_tri_aa_nb_sample_x2_diff_crop/2025-05-08_19-10-51/epoch=75-step=107616/run_2025-05-10_22-53-09'

for pdb in os.listdir(my_dir):
    if not '.' in pdb:
        pdb_path = os.path.join(my_dir, pdb)
        if not os.path.isfile(gt_pdb_path):
            print(f"GT PDB not found for {pdb}")
            continue
    
        for sample in os.listdir(pdb_path):
            sample_path = os.path.join(pdb_path, sample, f"cdr_rmsd.json")

            output_pdb_dir = os.path.join(output_dir, pdb)
            if not os.path.exists(output_pdb_dir):
                os.makedirs(output_pdb_dir, exist_ok=True)
            output_path = os.path.join(output_pdb_dir, f'my_{sample}.json')
            shutil.copy(sample_path, output_path)